# Embeddings

Loads the same preprocessed GoEmotions dataset as Project 1 (via `projects/src/data.py`).

In [5]:
# Setup: make the course code (part1) and our shared code (projects/src) importable.
# Run from projects/project2 (VS Code / JupyterLab do this by default).
import sys
from pathlib import Path

PROJECT2_DIR = Path.cwd().resolve()
REPO_ROOT = PROJECT2_DIR.parents[1]
PROJECTS_DIR = REPO_ROOT / "projects"
for p in (REPO_ROOT, PROJECTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import pandas as pd

from src import config, data

## Load preprocessed dataset

In [7]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)

df = pd.DataFrame(pairs, columns=["text", "label"])
df["label_name"] = df["label"].map(outcome_space.name)
print(len(df), outcome_space)
df.head()

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}
45446 0: admiration, 1: amusement, 2: anger, 3: annoyance, 4: approval, 5: caring, 6: confusion, 7: curiosity, 8: desire, 9: disappointment, 10: disapproval, 11: disgust, 12: embarrassment, 13: excitement, 14: fear, 15: gratitude, 16: grief, 17: joy, 18: love, 19: nervousness, 20: optimism, 21: pride, 22: realization, 23: relief, 24: remorse, 25: sadness, 26: surprise, 27: neutral


,text,label,label_name
0,Fast as [NAME] will carry me. Seriously uptown...,4,approval
1,You blew it. They played you like a fiddle.,27,neutral
2,TL;DR No more Superbowls for [NAME]. Get ready...,9,disappointment
3,So much time saved. Not.,7,curiosity
4,Emotes have a ridiculous amount of effort put ...,27,neutral


## Pretrained embeddings: GloVe Twitter (27B tokens, 1.2M vocab)

The downloaded zip holds four files (25d / 50d / 100d / 200d), but `part2.clslib.PretrainedEmbeddings` only accepts a zip with exactly one file. So we extract the size we want once into `data/cache/` (git-ignored) and load the `.txt` from there.

In [8]:
import zipfile

import part2.clslib

GLOVE_ZIP = PROJECTS_DIR / "src" / "embedders" / "glove.twitter.27B.zip"

  # GloVe Twitter zip with all 4 sizes
CACHE_DIR = PROJECT2_DIR / "data" / "cache"                  # git-ignored


def download_pretrained(dims=200):
    """
    Like the tutorial's download_pretrained, but for GloVe Twitter.
    Extracts glove.twitter.27B.{dims}d.txt from GLOVE_ZIP (skipped if already there).
    Returns (path, dims, skip_first_line) for PretrainedEmbeddings.
    """
    if dims not in (25, 50, 100, 200):
        raise ValueError(f"GloVe Twitter has 25/50/100/200 dims, not {dims}")
    name = f"glove.twitter.27B.{dims}d.txt"
    destination = CACHE_DIR / name
    if not destination.is_file():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(GLOVE_ZIP) as archive:
            archive.extract(name, CACHE_DIR)
    return str(destination), dims, False   # GloVe files have no header line

In [9]:
%%time
embtable_version, embtable_dims, embtable_metadata = download_pretrained(200)
embtable_version, embtable_dims, embtable_metadata

CPU times: user 6.2 s, sys: 750 ms, total: 6.95 s
Wall time: 7.3 s


('/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/projects/project2/data/cache/glove.twitter.27B.200d.txt',
 200,
 False)

In [10]:
%%time
# a few minutes for the 200d file (1.2M words)
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

CPU times: user 1min 10s, sys: 1.41 s, total: 1min 12s
Wall time: 1min 13s


In [11]:
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}")
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1193515, 200)
Size of vocabulary: 1193515
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:<user>
3:.
4::
5:rt
6:,
7:<repeat>
8:<hashtag>
9:<number>


## MLP classifier (embed-and-pool → MLP → log-softmax)

### Tokenizer

In [12]:
from collections import Counter
from part2 import clslib
import nltk

nltk.download("punkt_tab", quiet=True)

tokenizer = clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
)

# Check how the tokenizer handles Reddit comments
for text, label in pairs[:5]:
    print("Original:", text)
    print("Tokens:  ", tokenizer.tokens(text))
    print()

Original: Fast as [NAME] will carry me. Seriously uptown to downtown via Claiborne or broad isn’t that bad
Tokens:   ['fast', 'as', '[', 'name', ']', 'will', 'carry', 'me', '.', 'seriously', 'uptown', 'to', 'downtown', 'via', 'claiborne', 'or', 'broad', 'isn', '’', 't', 'that', 'bad']

Original: You blew it. They played you like a fiddle.
Tokens:   ['you', 'blew', 'it', '.', 'they', 'played', 'you', 'like', 'a', 'fiddle', '.']

Original: TL;DR No more Superbowls for [NAME]. Get ready for another winning season that ends in disappointment.
Tokens:   ['tl', ';', 'dr', 'no', 'more', 'superbowls', 'for', '[', 'name', ']', '.', 'get', 'ready', 'for', 'another', 'winning', 'season', 'that', 'ends', 'in', 'disappointment', '.']

Original: So much time saved. Not.
Tokens:   ['so', 'much', 'time', 'saved', '.', 'not', '.']

Original: Emotes have a ridiculous amount of effort put into them, which only makes the fact that nobody uses them sadder
Tokens:   ['emotes', 'have', 'a', 'ridiculous', 'am

In [13]:
token_counts = Counter()

for text, label in pairs:
    token_counts.update(tokenizer.tokens(text))

vocab = set(token_counts)

print("Unique tokens:", len(vocab))
print("Total token occurrences:", sum(token_counts.values()))
print("\nMost frequent tokens:")
print(token_counts.most_common(20))

Unique tokens: 29784
Total token occurrences: 697506

Most frequent tokens:
[('.', 37371), ('i', 20475), ('the', 18563), (',', 14413), ('to', 12961), ('a', 12831), ('you', 11402), ('it', 10959), ('that', 9935), ('and', 9227), ('is', 9022), ('!', 8806), ('name', 8489), ('[', 8426), (']', 8424), ('of', 7398), ('’', 6819), ('this', 6591), ('in', 6088), ('for', 5789)]


In [19]:
# Unknown tokens map to the default vocabulary index
unk_idx = trained_emb_model.vocab.default_index

def has_embedding(token):
    return trained_emb_model.vocab.index(token) != unk_idx

# Count how many unique tokens have embeddings
covered_vocab = {
    token for token in vocab
    if has_embedding(token)
}

type_coverage = len(covered_vocab) / len(vocab)

# Frequency-weighted coverage
total_tokens = sum(token_counts.values())
covered_tokens = sum(
    count for token, count in token_counts.items()
    if has_embedding(token)
)

token_coverage = covered_tokens / total_tokens

print(f"Type coverage:  {type_coverage:.2%}")
print(f"Token coverage: {token_coverage:.2%}")

# Most frequent missing tokens
unknown = [
    (token, count)
    for token, count in token_counts.most_common()
    if not has_embedding(token)
]

print("\nMost frequent OOV tokens:")
print(unknown[:30])

Type coverage:  79.47%
Token coverage: 97.97%

Most frequent OOV tokens:
[('...', 2194), ('..', 434), ('....', 348), ('2', 336), ('3', 300), ('1', 200), ('5', 188), ('4', 130), ('10', 129), ('/s', 123), ('100', 110), ('6', 109), ('.....', 97), ('20', 95), ('😂', 75), ('30', 71), ('15', 69), ('7', 66), ('8', 65), ('downvoted', 57), ('2nd', 50), ('9', 44), ('40', 43), ('downvote', 43), ('2019', 40), ("y'all", 38), ('90', 37), ('50', 37), ('25', 37), ('12', 36)]


In [28]:
# semantic similarity between words
import numpy as np

def get_vector(word):
    idx = trained_emb_model.vocab.index(word)

    if idx == trained_emb_model.vocab.default_index:
        return None

    return np.asarray(
        trained_emb_model.vectors[idx],
        dtype=np.float32
    )


def cosine_similarity(word1, word2):
    v1 = get_vector(word1)
    v2 = get_vector(word2)

    if v1 is None or v2 is None:
        return None

    return float(
        np.dot(v1, v2) /
        (np.linalg.norm(v1) * np.linalg.norm(v2))
    )


word_pairs = [
    ("happy", "joyful"),
    ("angry", "furious"),
    ("sad", "depressed"),
    ("grateful", "thankful"),
    ("happy", "angry"),
    ("love", "hate"),
    ("good", "bad"),
]

for w1, w2 in word_pairs:
    similarity = cosine_similarity(w1, w2)

    if similarity is None:
        print(f"{w1:12} {w2:12} OOV")
    else:
        print(f"{w1:12} {w2:12} {similarity:.4f}")

happy        joyful       0.3519
angry        furious      0.3684
sad          depressed    0.7040
grateful     thankful     0.8743
happy        angry        0.4547
love         hate         0.6842
good         bad          0.7984


### Encoder (average of word embeddings)

In [29]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

tokenizer.vocab = trained_emb_model.vocab

pad_idx = trained_emb_model.vocab.index(
    trained_emb_model.pad_token
)

embedding_matrix = torch.as_tensor(
    trained_emb_model.vectors,
    dtype=torch.float32
)

encoder = part2.clslib.EmbedAndPoolEncoder(
    embedding_matrix=embedding_matrix,
    freeze_embeddings=True,
    padding_idx=pad_idx,
)

In [30]:
texts = [text for text, label in pairs[:4]]

sequences = [
    torch.tensor(tokenizer.ids(text), dtype=torch.long)
    for text in texts
]

batch = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=pad_idx,
)

with torch.no_grad():
    representations = encoder(batch)

print("Input shape:", batch.shape)
print("Encoder output shape:", representations.shape)

Input shape: torch.Size([4, 22])
Encoder output shape: torch.Size([4, 200])


### Model (MLP on top of the encoder)

In [31]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    """One-hidden-layer MLP, copied from part1.smlp.SparseMLP.

    Inputs: feature vectors of size `input_dim` (here: the pooled embedding, e.g. 200d GloVe).
    Process: linear into a hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.

    Only change vs. part1: the first layer uses a dense matmul (x @ W) instead of
    torch.sparse.mm, since pooled embeddings are dense vectors.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. embedding dimension)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )
        self.bias = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.output = nn.Sequential(
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1) # for numerical stability, it is common to use log softmax, rather than softmax
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        """Map a batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        x = x @ self.weight
        x = x + self.bias

        x = self.activation(x)
        x = self.dropout(x)
        # logprobs
        return self.output(x)


### Cross-validation splits

In [32]:
from part1 import experiment as p1_experiment

# Exactly the folds from Project 1 (02_experiments.ipynb): same pairs, same split function, same k and seed.
# Same folds = we can compare Project 1 and Project 2 models fold by fold (paired comparison).
folds = p1_experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

for fold, split in folds.items():
    print(f"fold {fold}: {len(split['training'])} training, {len(split['validation'])} validation")

fold 0: 36356 training, 9090 validation
fold 1: 36357 training, 9089 validation
fold 2: 36357 training, 9089 validation
fold 3: 36357 training, 9089 validation
fold 4: 36357 training, 9089 validation


### Training

In [33]:
import json
from collections import OrderedDict

import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import part2.experiment
from src import models   # Project 1 helpers: resolve_device, count_parameters, epoch_table, summary_row

# Same settings as Project 1's one-hidden-layer MLP (02_experiments.ipynb smlp_args + part1.smlp defaults)
mlp_args = dict(hidden_dim=128, dropout=0.1)
train_args = dict(num_epochs=10, batch_size=64, learning_rate=1e-3, weight_decay=1e-4)
device = models.resolve_device("auto")   # CUDA if available, else CPU (same as Project 1)

RESULTS_DIR = PROJECT2_DIR / "results"   # same JSON format as Project 1's results/
MODELS_DIR = PROJECT2_DIR / "models"     # git-ignored (projects/*/models/)

# Tokenize every comment once; all folds reuse these ids
ids_of = {text: tokenizer.ids(text) for text, _ in tqdm(pairs, desc="tokenizing")}


def pool_encoder(freeze=True):
    """A fresh embed-and-pool encoder. Trainable embeddings get their own copy, so folds don't leak into each other."""
    matrix = embedding_matrix if freeze else embedding_matrix.clone()
    return part2.clslib.EmbedAndPoolEncoder(matrix, padding_idx=pad_idx, freeze_embeddings=freeze)


def make_loader(split, shuffle):
    dataset = part2.experiment.LabelledCorpus([ids_of[t] for t, _ in split], [int(y) for _, y in split])
    return DataLoader(dataset, batch_size=train_args["batch_size"], shuffle=shuffle,
                      collate_fn=lambda b: part2.experiment.collate_labelled_texts(b, padding_value=pad_idx))


def train_fold(make_encoder, training_set, validation_set, save_to):
    """
    One fold, trained like Project 1 (part1.smlp): AdamW + NLLLoss for num_epochs, keep the epoch
    with the lowest validation loss and report its validation accuracy.
    Model = encoder (text ids -> one vector) followed by our MLP (vector -> log-probs).
    """
    torch.manual_seed(config.SEED)
    enc = make_encoder()
    model = nn.Sequential(enc, MLP(input_dim=enc.enc_dim, num_classes=len(outcome_space), **mlp_args)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_args["learning_rate"],
                                  weight_decay=train_args["weight_decay"])
    criterion = nn.NLLLoss()
    train_loader, val_loader = make_loader(training_set, True), make_loader(validation_set, False)

    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    metrics, best_epoch, best_state = [], None, None
    for epoch in range(train_args["num_epochs"]):
        train_metrics = part2.experiment.training_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = part2.experiment.evaluate(model, val_loader, criterion, device)
        metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})
        if best_epoch is None or val_metrics["loss"] <= metrics[best_epoch]["val_metrics"]["loss"]:
            best_epoch = epoch
            # only the trained weights (the frozen 1.2M x 200 GloVe matrix would be ~1 GB per fold)
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items() if n in trainable}

    save_to.parent.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, f"{save_to}.pt")   # reload with model.load_state_dict(state, strict=False)
    preds = np.concatenate(metrics[best_epoch]["val_metrics"]["predictions"])
    return metrics, best_epoch, preds, models.count_parameters(model)


def cross_validate(name, make_encoder):
    """k-fold CV over Project 1's folds. Returns the same dict as Project 1's models.run_experiments."""
    out = {k: OrderedDict() for k in ("models", "predictions", "accuracies", "curves", "best_epochs")}
    for fold, split in tqdm(folds.items(), desc=name):
        save_to = MODELS_DIR / name / f"fold-{fold}" / "model"
        metrics, best_epoch, preds, n_params = train_fold(make_encoder, split["training"], split["validation"], save_to)
        out["models"][fold] = str(save_to)
        out["predictions"][fold] = preds
        out["accuracies"][fold] = metrics[best_epoch]["val_metrics"]["accuracy"]
        out["curves"][fold] = [{s: {m: ep[s][m] for m in ("loss", "accuracy")} for s in ep} for ep in metrics]
        out["best_epochs"][fold] = best_epoch
    out["n_params"] = n_params
    return out


def save_results(name, out, **meta):
    """Same JSON as Project 1's results_io.save_cv, but written to projects/project2/results/."""
    record = {
        "name": name,
        "accuracies": {str(f): float(a) for f, a in out["accuracies"].items()},
        "meta": meta,
        "n_params": out["n_params"],
        "best_epochs": {str(f): int(e) for f, e in out["best_epochs"].items()},
        "curves": {str(f): c for f, c in out["curves"].items()},
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f"{name}.json"
    path.write_text(json.dumps(record, indent=2))
    return path

tokenizing:   0%|          | 0/45446 [00:00<?, ?it/s]

In [34]:
%%time
# Embed-and-pool (frozen GloVe Twitter) + Project 1's MLP, on Project 1's folds
name = f"mlp__glove{embtable_dims}-pool"
out = cross_validate(name, lambda: pool_encoder(freeze=True))
save_results(name, out, embeddings=f"glove.twitter.27B.{embtable_dims}d", freeze_embeddings=True,
             tokenizer="BasicTokenizer(lowercase, keep punct, keep stopwords)", **mlp_args, **train_args)

acc = np.array(list(out["accuracies"].values()))
print(f"{name}: CV accuracy {acc.mean():.4f} ± {acc.std():.4f}  (mean ± std over {len(acc)} folds)")
models.epoch_table(out)

mlp__glove200-pool:   0%|          | 0/5 [00:00<?, ?it/s]

CPU times: total: 4min 48s
Wall time: 44.1 s


KeyboardInterrupt: 

### Evaluation